##### Axis 2 with Opphold
**Step 1: Pick only axix one from `https://finnkode.helsedirektoratet.no/phbu/chapter?q=4` and exclude `"2000", "2999", "999", "000"` and date from `1995-01-01`, `save all the axis 2 patient record`` and `axis 2 patient with primary diagnosis`**

In [ ]:
import os
import sys

parent_dir = os.path.join(os.path.dirname(os.path.abspath("")), os.pardir)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from importall import *

SRC_PARQUET = os.getenv("MAIN_DATASET")
DST_PRIMARY = os.getenv("AXISII_MAIN_DATASET")

df = dd.read_parquet(SRC_PARQUET, engine="pyarrow")
cutoff = "1995-01-01"
df["sak_igangdato"] = dd.to_datetime(df["sak_igangdato"], errors="coerce")
df["sak_avsldato"] = dd.to_datetime(df["sak_avsldato"], errors="coerce")

df = df[(df["sak_igangdato"] >= cutoff) & (df["sak_avsldato"] >= cutoff)]
df = df[(df["patient_age"] >= 1) & (df["patient_age"] <= 19)]
df = df[df["diag_akse"] == "2"]

print(
    "Unique patients • age 1-19 • Axis 2 • episodes ≥ 1995:",
    df["pasient_nr"].nunique().compute(),
)

df_primary = (
    df.fillna({"diag_hoved": "0"})
    .query("diag_hoved == '1'")
    .dropna(subset=["diag_diagnose", "atckode"])
    .loc[lambda d: ~d["diag_diagnose"].isin(["2000", "2999", "999", "000"])]
)
zeros_before = (
    df_primary[df_primary["diag_diagnose"] == "000"]["pasient_nr"].nunique().compute()
)
print(f"Patients with diag_diagnose '000' before exclude: {zeros_before}")


print(
    "Unique patients • all of the above + medicated:",
    df_primary["pasient_nr"].nunique().compute(),
)


DST_PRIMARY.mkdir(parents=True, exist_ok=True)

(
    df_primary.repartition(npartitions=1).to_parquet(
        DST_PRIMARY, engine="pyarrow", write_index=False, write_metadata_file=False
    )
)
print("Parquet files written:")
print(f" {DST_PRIMARY}")

##### Start here
**Step 1(a): Research Question: 1. Overview of BUP data in axis 2 after above filter?**
***Read parquet file with Dask***

In [ ]:
import os
import sys

parent_dir = os.path.join(os.path.dirname(os.path.abspath("")), os.pardir)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from importall import *

output_file19 = os.getenv("AXISII_MAIN_DATASET")
df_axis_one_19 = dd.read_parquet(output_file19, engine="pyarrow")

print("Shape:", df_axis_one_19.shape)
print(
    "Unique cases under 19 with diagnose akse 2 from 1995:",
    df_axis_one_19["sak_nr"].nunique().compute(),
)
print(
    "Unique patients under 19 with diagnose akse 2 from 1995:",
    df_axis_one_19["pasient_nr"].nunique().compute(),
)
print("Oldest sak_igangdato:", df_axis_one_19["sak_igangdato"].min().compute())
print("Newest sak_igangdato:", df_axis_one_19["sak_igangdato"].max().compute())
print("Oldest sak_avsldato:", df_axis_one_19["sak_avsldato"].min().compute())
print("Newest sak_avsldato:", df_axis_one_19["sak_avsldato"].max().compute())
print(
    "Unique diagnose code under 19 with diagnose akse 2 from 1995:",
    df_axis_one_19["diag_diagnose"].nunique().compute(),
)
print(
    "Unique ATC code under 19 with diagnose akse 2 from 1995:",
    df_axis_one_19["atckode"].nunique().compute(),
)
print(
    f"Unique cases under 19 with axis 2 from 1995: {df_axis_one_19["sak_nr"].nunique().compute()} and {df_axis_one_19["opphold_id"].nunique().compute()}"
)

**Extra: What are the most frequent medication and diagnosis across all the diagnoses and across the primary or principle diagnosis for the patient across Axis 2?**

In [ ]:
import os
import sys

parent_dir = os.path.join(os.path.dirname(os.path.abspath("")), os.pardir)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from importall import *
from updated_mapped_icd_codes import mapped_icd_codes

output_plot = os.getenv("OUTPUT_PLOTS")

with open(os.getenv("ATCNAME_CODE_NORSK") + "atcnavn_code.json", "r") as f:
    atc_mapping = json.load(f)
atc_map = {code: info.get("atcnavn", "Unknown") for code, info in atc_mapping.items()}
input_path = os.getenv("AXISII_MAIN_DATASET")
df = dd.read_parquet(input_path, engine="pyarrow")

# top 20 diagnosis and medication in axis 1
diag_list = [
    "F810",
    "F813",
    "F83",
    "F801",
    "F82",
    "F819",
    "F802",
    "F809",
    "F812",
    "F811",
    "F800",
    "F808",
    "F818",
    "F80",
    "F89",
    "F88",
    "F81",
]
med_list = [
    "N06BA04",
    "A06BA04",
    "N06BA09",
    "N06BA12",
    "N05CH01",
    "N06AB06",
    "N05AX08",
    "N06AB03",
    "R06AD01",
    "N05AX12",
    "N05AH04",
    "N03AX09",
    "N06BA02",
    "N05CF01",
    "A11EA",
    "N05AA02",
    "D07BC01",
    "N05CD02",
    "N05AH03",
    "H01BA02",
]

diag_counts = (
    df[df["diag_diagnose"].isin(diag_list)]
    .groupby("diag_diagnose")["pasient_nr"]
    .nunique()
    .compute()
    .sort_values(ascending=False)
)

med_counts = (
    df[df["atckode"].isin(med_list)]
    .groupby("atckode")["pasient_nr"]
    .nunique()
    .compute()
    .sort_values(ascending=False)
)

diag_labels = [
    f"{code} – {mapped_icd_codes.get(code,'Unknown')}" for code in diag_counts.index
]
med_labels = [f"{code} – {atc_map.get(code,'Unknown')}" for code in med_counts.index]

wrapped_diag_labels = [textwrap.fill(lbl, width=30) for lbl in diag_labels]
wrapped_med_labels = [textwrap.fill(lbl, width=30) for lbl in med_labels]

# ─── plot diagnoses
width = 0.6
x = np.arange(len(diag_counts))

fig, ax = plt.subplots(figsize=(16, 8))
bars = ax.bar(x, diag_counts.values, width=width, color="#0072B2", alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels(wrapped_diag_labels, rotation=90)
ax.set_xlabel("Diagnosis (ICD-10 code – description)")
ax.set_ylabel("Unique Patient Count")
ax.set_title("Axis 2 -  Most Common Primary Diagnosis")
ax.bar_label(bars, labels=[f"{v:,}" for v in diag_counts.values], padding=2, fontsize=8)
plt.tight_layout()
# plt.savefig(output_plot + "axis1/axis1_1(b)part1.png", dpi=400, bbox_inches="tight")
plt.show()

# ─── plot medications
x = np.arange(len(med_counts))

fig, ax = plt.subplots(figsize=(16, 8))
bars = ax.bar(x, med_counts.values, width=width, color="#D55E00", alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels(wrapped_med_labels, rotation=90)
ax.set_xlabel("Medication (ATC code – name)")
ax.set_ylabel("Unique Patient Count")
# ax.set_title("Axis 2 -  Most Common Medication")
ax.bar_label(bars, labels=[f"{v:,}" for v in med_counts.values], padding=2, fontsize=8)
plt.tight_layout()
# plt.savefig(output_plot + "axis1/axis1_1(b)part2.png", dpi=400, bbox_inches="tight")
plt.show()